# Group Master Total - Production Calibrated

Production workflow with:
- Markov lifecycle and configurable DEL90 K source
- Portfolio-level true-as-of DEL90 calibration by forecast anchor MOB
- Cohort reconciliation and deterministic loan ranking
- Group cache, lifecycle report, loan report, and master workbook

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rollrate.group_master_cache import (
    build_master_total_from_staging,
    group_stage_dir,
    list_staged_groups,
    load_frame,
    normalize_run_cfg,
    recalibrate_selected_groups_from_stage,
    run_selected_groups,
)

## Production Configuration

In [ ]:
DEL90_MOB12_CALIBRATION = {
    'del90_k_source': 'blend',
    'del90_blend_anchor_mobs': [2, 4, 6, 8],
    'del90_blend_weight_grid': [round(step * 0.05, 2) for step in range(21)],
    'del90_blend_n_vintages': 6,
    'del90_blend_min_vintages': 4,
    'del90_blend_half_life_months': 3.0,
    'del90_blend_fallback_weight': 1.0,
    'del90_portfolio_calibration_enabled': False,
    'del90_calibration_anchor_mobs': [6, 8],
    'del90_calibration_n_vintages': 6,
    'del90_calibration_min_vintages': 4,
    'del90_calibration_half_life_months': 3.0,
    'del90_calibration_min_disb': 1.0,
    'del90_calibration_shrink': 0.5,
    'del90_calibration_shrink_by_anchor': {2: 1.0, 4: 0.5, 6: 0.5, 8: 0.5},
    'del90_calibration_residual_cap': 0.05,
    'del90_calibration_enforce_del30_cap': True,
    'del90_calibration_mae_guardrail': True,
    'del90_calibration_drift_warning': 0.01,
    'del90_anchor_source_by_anchor': {
        5: 'del90',
        6: 'del90',
        7: 'del90',
        8: 'del90',
        9: 'del90',
        10: 'del90',
        11: 'del90',
    },
    'del90_proxy_source_by_anchor': {
        3: 'del30',
        4: 'del30',
    },
    'del90_proxy_n_vintages': 6,
    'del90_proxy_min_vintages': 4,
    'del90_proxy_half_life_months': 3.0,
    'del90_proxy_ridge_grid': [0.0, 0.01, 0.1, 1.0, 5.0, 10.0, 25.0],
    'del90_proxy_mae_guardrail': True,
    'del90_proxy_enforce_del30_cap': True,
}

GROUP_RUNS = [
    {
        'name': 'POS',
        'data_path': r'//hcm-filesrv.mafc.vn/Risk/PORTFOLIO/Management report/Projection/Data/POS_Parquet_MSCORE_V2',
        'max_mob': 24,
        'target_mobs': [12],
        'group_portfolio_name': 'TOTAL_POS',
        'segment_cols': ['PRODUCT_TYPE', 'RISK_SCORE', 'SALE_CHANNEL', 'GENDER'],
        'loan_min_vintage': '2023-07-01',
        'loan_base_mode': 'latest_cutoff',
        'run_allocation': True,
        'export_group_workbook': True,
        'export_loan_forecast': True,
        **DEL90_MOB12_CALIBRATION,
    },
    # {
    #     'name': 'NTB',
    #     'data_path': r'D:/backup_? c/NTB_Parquet_NTBV3_NEWRG',
    #     'max_mob': 24,
    #     'target_mobs': [12],
    #     'group_portfolio_name': 'TOTAL_NTB',
    #     'segment_cols': ['PRODUCT_TYPE', 'RISK_SCORE', 'SALE_CHANNEL', 'GENDER'],
    #     'loan_min_vintage': '2023-07-01',
    #     'loan_base_mode': 'latest_per_loan',
    #     'run_allocation': True,
    #     'export_group_workbook': True,
    #     'export_loan_forecast': True,
    #     **DEL90_MOB12_CALIBRATION,
    # },
    # {
    #     'name': 'ETB',
    #     'data_path': r'//hcm-filesrv.mafc.vn/Risk/PORTFOLIO/Management report/Projection/Data/ETB_Parquet',
    #     'max_mob': 24,
    #     'target_mobs': [12],
    #     'group_portfolio_name': 'TOTAL_ETB',
    #     'segment_cols': ['PRODUCT_TYPE', 'RISK_SCORE', 'SALE_CHANNEL', 'GENDER'],
    #     'loan_min_vintage': '2023-07-01',
    #     'loan_base_mode': 'latest_per_loan',
    #     'run_allocation': True,
    #     'export_group_workbook': True,
    #     'export_loan_forecast': True,
    #     **DEL90_MOB12_CALIBRATION,
    # },
]


In [ ]:
OUTPUT_ROOT = project_root / 'outputs' / 'production_calibrated'
STAGING_ROOT = project_root / 'staging' / 'production_calibrated'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

ACTIVE_GROUPS = None
FORCE_GROUPS = None
SKIP_EXISTING_STAGE = False
SAVE_FULL_GROUP_CACHE = True
LOAD_FROM_STAGING_ONLY = False
FAST_RECALIBRATE_FROM_STAGE = False

MASTER_PORTFOLIO_NAME = 'MASTER_TOTAL'
USE_ALL_STAGED_GROUPS_FOR_MASTER = False
MASTER_INCLUDE_GROUPS = None
STRICT_CONFIG_MATCH_FOR_MASTER = True

display(pd.DataFrame([{
    'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'STAGING_ROOT': str(STAGING_ROOT),
    'ACTIVE_GROUPS': ACTIVE_GROUPS,
    'FORCE_GROUPS': FORCE_GROUPS,
    'SKIP_EXISTING_STAGE': SKIP_EXISTING_STAGE,
    'SAVE_FULL_GROUP_CACHE': SAVE_FULL_GROUP_CACHE,
    'LOAD_FROM_STAGING_ONLY': LOAD_FROM_STAGING_ONLY,
    'FAST_RECALIBRATE_FROM_STAGE': FAST_RECALIBRATE_FROM_STAGE,
    'MASTER_PORTFOLIO_NAME': MASTER_PORTFOLIO_NAME,
    'USE_ALL_STAGED_GROUPS_FOR_MASTER': USE_ALL_STAGED_GROUPS_FOR_MASTER,
    'STRICT_CONFIG_MATCH_FOR_MASTER': STRICT_CONFIG_MATCH_FOR_MASTER,
}]))


## Run Groups

In [ ]:
run_summary_df = pd.DataFrame()

if LOAD_FROM_STAGING_ONLY:
    print('Skip group execution. Using staging only.')
elif FAST_RECALIBRATE_FROM_STAGE:
    run_summary_df = recalibrate_selected_groups_from_stage(
        GROUP_RUNS,
        output_root=OUTPUT_ROOT,
        staging_root=STAGING_ROOT,
        selected_groups=ACTIVE_GROUPS,
        force_groups=FORCE_GROUPS,
    )
    display(run_summary_df)
else:
    run_summary_df = run_selected_groups(
        GROUP_RUNS,
        output_root=OUTPUT_ROOT,
        staging_root=STAGING_ROOT,
        selected_groups=ACTIVE_GROUPS,
        skip_existing_stage=SKIP_EXISTING_STAGE,
        force_groups=FORCE_GROUPS,
        save_full_cache=SAVE_FULL_GROUP_CACHE,
    )
    display(run_summary_df)

## Calibration Audit

In [ ]:
calibration_audit = {}
proxy_audit = {}
k_curve_audit = {}

for raw_cfg in GROUP_RUNS:
    cfg = normalize_run_cfg(raw_cfg)
    if ACTIVE_GROUPS is not None and cfg['name'] not in set(ACTIVE_GROUPS):
        continue
    stage_dir = group_stage_dir(STAGING_ROOT, cfg['name'])
    try:
        curve = load_frame(stage_dir / 'del90_calibration_curve')
    except FileNotFoundError:
        curve = pd.DataFrame()
    if not curve.empty:
        calibration_audit[cfg['name']] = curve
        print(f"\n{cfg['name']} calibration curve")
        display(curve)

    try:
        proxy_curve = load_frame(stage_dir / 'del90_proxy_curve')
    except FileNotFoundError:
        proxy_curve = pd.DataFrame()
    if not proxy_curve.empty:
        proxy_audit[cfg['name']] = proxy_curve
        print(f"\n{cfg['name']} DEL90 proxy curve")
        display(proxy_curve)

    try:
        k_curve = load_frame(stage_dir / 'k_curve')
    except FileNotFoundError:
        k_curve = pd.DataFrame()
    if not k_curve.empty:
        k_curve_audit[cfg['name']] = k_curve
        print(f"\n{cfg['name']} K curves")
        display(k_curve)

if not calibration_audit and not proxy_audit:
    print('No calibration/proxy curves found in staging.')


## Loan And Lifecycle Validation

In [ ]:
validation_rows = []

for raw_cfg in GROUP_RUNS:
    cfg = normalize_run_cfg(raw_cfg)
    if ACTIVE_GROUPS is not None and cfg['name'] not in set(ACTIVE_GROUPS):
        continue
    stage_dir = group_stage_dir(STAGING_ROOT, cfg['name'])
    try:
        lifecycle = load_frame(stage_dir / 'lifecycle_final')
    except FileNotFoundError:
        continue

    for mob in cfg['target_mobs']:
        target_rows = lifecycle[lifecycle['MOB'] == mob].copy()
        forecast_rows = target_rows[target_rows['IS_FORECAST'] == 1].copy()
        validation_rows.append({
            'GROUP': cfg['name'],
            'TARGET_MOB': mob,
            'FORECAST_COHORT_ROWS': len(forecast_rows),
            'CALIBRATED_ROWS': int(forecast_rows.get('DEL90_CAL_APPLIED', pd.Series(dtype=int)).sum()),
            'ROUTED_ROWS': int(forecast_rows.get('DEL90_ROUTE_APPLIED', pd.Series(dtype=int)).sum()),
            'PROXY_ROWS': int(forecast_rows.get('DEL90_PROXY_APPLIED', pd.Series(dtype=int)).sum()),
            'MAX_DEL90_MINUS_DEL30': (
                float((forecast_rows['DEL90_PCT'] - forecast_rows['DEL30_PCT']).max())
                if not forecast_rows.empty else 0.0
            ),
            'DEL90_PCT_MIN': float(forecast_rows['DEL90_PCT'].min()) if not forecast_rows.empty else None,
            'DEL90_PCT_MAX': float(forecast_rows['DEL90_PCT'].max()) if not forecast_rows.empty else None,
        })

    try:
        loans = load_frame(stage_dir / 'loan_forecast')
    except FileNotFoundError:
        loans = pd.DataFrame()

    if not loans.empty:
        for mob in cfg['target_mobs']:
            prob_col = f'PROB_DEL90_MOB{mob}'
            raw_prob_col = f'PROB_DEL90_RAW_MOB{mob}'
            ead_col = f'EAD_DEL90_MOB{mob}'
            flag_col = f'DEL90_FLAG_MOB{mob}'
            if prob_col in loans:
                print(f"\n{cfg['name']} loan allocation MOB{mob}")
                display(pd.DataFrame([{
                    'LOANS': len(loans),
                    'AVG_RAW_PROB': loans[raw_prob_col].mean() if raw_prob_col in loans else None,
                    'AVG_CALIBRATED_PROB': loans[prob_col].mean(),
                    'EAD_DEL90': loans[ead_col].sum() if ead_col in loans else None,
                    'SELECTED_DEL90_LOANS': int(loans[flag_col].sum()) if flag_col in loans else None,
                }]))

validation_df = pd.DataFrame(validation_rows)
display(validation_df)


## Build Master Workbook

In [ ]:
master_result = build_master_total_from_staging(
    GROUP_RUNS,
    MASTER_PORTFOLIO_NAME,
    output_root=OUTPUT_ROOT,
    staging_root=STAGING_ROOT,
    use_all_staged_groups=USE_ALL_STAGED_GROUPS_FOR_MASTER,
    include_groups=MASTER_INCLUDE_GROUPS,
    strict_config_match=STRICT_CONFIG_MATCH_FOR_MASTER,
)

print('Master workbook:', master_result['master_file'])
print('Groups used:', master_result['groups_used'])
display(master_result['summary_df'])
display(master_result['coverage'])